In [1]:
import pandas as pd

Website for downloading data: https://data.marine.copernicus.eu/product/INSITU_BAL_PHYBGCWAV_DISCRETE_MYNRT_013_032/download?dataset=cmems_obs-ins_bal_phybgcwav_mynrt_na_irr_202311--ext--history

In [2]:
times = pd.date_range(start="2021-01-01", end="2025-12-31", freq='30min')

In [3]:
def prepare_dfs(df):
    # Since it is timezone UTC, we can safely remove the 'Z' and 'T' characters and convert to datetime
    df["datetime_UTC"]=pd.to_datetime(df['time'].str.replace('T', ' ').str.replace('Z', ''))
    # Only choose every 30 minute data. It is different how frequent the data is, hence the funny looking indexing
    df_30min = df[df['datetime_UTC'].isin(times)]
    # Select only data with valueQc < 2 (good or probably good data)
    df_good = df_30min[df_30min['valueQc'] < 2]
    # Select only the columns we want to keep
    df_new = df_good.set_index('datetime_UTC')[['platformId', 'value']]
    # rename 'value' to 'SLEV' 
    df_new.rename(columns={'value': 'water_level'}, inplace=True)
    # Round values
    df_new['water_level'] = df_new['water_level'].round(4)
    # Now split df_new into separate dataframes for each platformId and collect them in a lookup dictionary
    platforms = df_new['platformId'].unique()
    dfs = {}
    for platform in platforms:
        dfs[platform] = df_new[df_new['platformId'] == platform]

    return dfs


In [4]:
files = ["cmems_obs_2020_2021.csv", "cmems_obs_2022_2023.csv", "cmems_obs_2024.csv", "cmems_obs_2025.csv"]

df =pd.read_csv(f"raw_data/{files[0]}", index_col=0, parse_dates=True)
dfs_all = prepare_dfs(df)

# Loop over files (from old to new) and add to dfs_all
for f in files[1:]:
    df =pd.read_csv(f"raw_data/{f}", index_col=0, parse_dates=True)
    dfs_next = prepare_dfs(df)

    # Concatenate the dataframes in dfs0 and dfs1 according to their keys, avoiding duplicated rows
    for key in dfs_all.keys():
        if key in dfs_next.keys():
            dfs_next[key] = dfs_next[key].loc[dfs_next[key].index > dfs_all[key].index[-1]]
            # Concatenate new data to existing data
            dfs_all[key] = pd.concat([dfs_all[key], dfs_next[key]])

    # If new stations are present, add them to the dictionary
    for key in dfs_next.keys():
        if key not in dfs_all.keys():
            dfs_all[key] = dfs_next[key]


In [5]:
# Check dfs_all for duplicate indices
for key in dfs_all.keys():
    if dfs_all[key].index.duplicated().any():
        print(f"Duplicate indices in {key}")

dfs_all.keys()

dict_keys(['Drogden', 'Klagshamn', 'Flinten7', 'Koege', 'Dragor', 'Skanor', 'MalmoHamn', 'Hornbaek', 'Kobenhavn', 'Viken', 'Vedbaek', 'Barseback', 'Helsingborg', 'Flinten16'])

In [6]:
# Go through the keys and save the dataframes to csv
# Add to the name the first present year in data and the last present year
skip_station = ["Viken","Flinten16"] # Skip this station since it is too far out of the domain
for key in dfs_all.keys():
    if key in skip_station:
        continue
    dfs_all[key].drop(columns=["platformId"]).to_csv(f"../observations/{key}_wl.csv", date_format='%Y-%m-%dT%H:%M:%S')

## Convert to dfs0

In [2]:
import os
import glob
import shutil
import mikeio
from mikeio import ItemInfo, EUMType, EUMUnit

# Exclude the station metadata file, only convert time series
csv_files = [
    f for f in sorted(glob.glob("../observations/*_wl.csv"))
    if os.path.basename(f) != "stations_wl.csv"
]

items = {"water_level": ItemInfo(EUMType.Water_Level, EUMUnit.meter)}

for csv_file in csv_files:
    df = pd.read_csv(csv_file, index_col="datetime_UTC", parse_dates=True)
    ds = mikeio.from_pandas(df, items=items)
    dfs0_file = os.path.splitext(csv_file)[0] + ".dfs0"
    ds.to_dfs(dfs0_file)
    print(f"Saved {dfs0_file}")


Saved ../observations\Barseback_wl.dfs0
Saved ../observations\Dragor_wl.dfs0
Saved ../observations\Drogden_wl.dfs0
Saved ../observations\Flinten7_wl.dfs0
Saved ../observations\Helsingborg_wl.dfs0
Saved ../observations\Hornbaek_wl.dfs0
Saved ../observations\Klagshamn_wl.dfs0
Saved ../observations\Kobenhavn_wl.dfs0
Saved ../observations\Koege_wl.dfs0
Saved ../observations\MalmoHamn_wl.dfs0
Saved ../observations\NordreRose_wl.dfs0
Saved ../observations\Skanor_wl.dfs0
Saved ../observations\Vedbaek_wl.dfs0


In [5]:
ds

<mikeio.Dataset>
dims: (time:85735)
time: 2021-01-01 00:00:00 - 2025-12-31 00:00:00 (85735 non-equidistant records)
geometry: GeometryUndefined()
items:
  0:  water_level <Water Level> (meter)

In [3]:
dest_dir = "raw_data/csv_versions"
os.makedirs(dest_dir, exist_ok=True)

for csv_file in csv_files:
    shutil.move(csv_file, os.path.join(dest_dir, os.path.basename(csv_file)))
    print(f"Moved {csv_file} -> {dest_dir}")


Moved ../observations\Barseback_wl.csv -> raw_data/csv_versions
Moved ../observations\Dragor_wl.csv -> raw_data/csv_versions
Moved ../observations\Drogden_wl.csv -> raw_data/csv_versions
Moved ../observations\Flinten7_wl.csv -> raw_data/csv_versions
Moved ../observations\Helsingborg_wl.csv -> raw_data/csv_versions
Moved ../observations\Hornbaek_wl.csv -> raw_data/csv_versions
Moved ../observations\Klagshamn_wl.csv -> raw_data/csv_versions
Moved ../observations\Kobenhavn_wl.csv -> raw_data/csv_versions
Moved ../observations\Koege_wl.csv -> raw_data/csv_versions
Moved ../observations\MalmoHamn_wl.csv -> raw_data/csv_versions
Moved ../observations\NordreRose_wl.csv -> raw_data/csv_versions
Moved ../observations\Skanor_wl.csv -> raw_data/csv_versions
Moved ../observations\Vedbaek_wl.csv -> raw_data/csv_versions
